# 1. Spark processing on EMR

## Setting up PySpark on an EMR Cluster

Before executing this Jupyter notebook, I launched a Spark-enabled AWS EMR cluster using the following command in Terminal. During the initial development process, I used `--core_count 2` with a smaller sample of the dataset for faster iteration and debugging. For the final analysis, あ increased the cluster size to improve scalability and performance.

```bash
python launch_spark_cluster.py --s3_bucket ***bucketnamehere** --primary_count 1 --core_count 2 --instance_type m5.xlarge
```

ex.
```bash
python launch_spark_cluster.py --s3_bucket nana-survey-bucket-2026 --primary_count 1 --core_count 2 --instance_type m5.xlarge
```


After the EMR cluster was launched, I configured SSH port forwarding to access JupyterHub running on the EMR primary node.

```bash
ssh -i ***.pem" -NL 9443:localhost:9443 hadoop@ec****.compute-1.amazonaws.com
```

ex.
```bash
ssh -i "C:\Users\evano\OneDrive\ドキュメント\GitHub\a4-NanaTakeshiba\labsuser.pem" -NL 9443:localhost:9443 hadoop@ec2-54-210-38-87.compute-1.amazonaws.com
```


I then navigated to `https://localhost:9443` in a web browser and executed this notebook through the remote JupyterHub environment connected to the Spark cluster.

In [1]:
%%configure -f
{
    "conf": {
        "spark.pyspark.python": "python3",
        "spark.pyspark.virtualenv.enabled": "true",
        "spark.pyspark.virtualenv.type":"native",
        "spark.pyspark.virtualenv.bin.path":"/usr/bin/virtualenv"
    }
}

In [2]:
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1779927251977_0002,pyspark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
# imports

import time

from pyspark.sql.functions import (
    col,
    trim,
    lower,
    regexp_replace,
    explode,
    array_remove,
    size
)

from pyspark.ml.feature import (
    Tokenizer,
    StopWordsRemover
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Loading JSON files

In [4]:
posts = spark.read.json(
    "s3://luchen-lab/raw/reddit/posts/source=archive/"
)

comments = spark.read.json(
    "s3://luchen-lab/raw/reddit/comments/source=archive/"
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Checking the style of data

In [5]:
print("Posts count:")
print(posts.count())

print("Comments count:")
print(comments.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Posts count:
1175
Comments count:
3810

In [6]:
print("Posts schema:")
posts.printSchema()

print("Comments schema:")
comments.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Posts schema:
root
 |-- created_date: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- month: integer (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- post_id: string (nullable = true)
 |-- score: long (nullable = true)
 |-- selftext: string (nullable = true)
 |-- source: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- title: string (nullable = true)
 |-- year: integer (nullable = true)

Comments schema:
root
 |-- body: string (nullable = true)
 |-- comment_id: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- month: integer (nullable = true)
 |-- post_id: string (nullable = true)
 |-- score: long (nullable = true)
 |-- source: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- year: integer (nullable = true)

In [7]:
posts.show(5, truncate=True)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+-----------+-----+------------+-------+-----+--------------------+-------+----------+--------------------+----+
|created_date|created_utc|month|num_comments|post_id|score|            selftext| source| subreddit|               title|year|
+------------+-----------+-----+------------+-------+-----+--------------------+-------+----------+--------------------+----+
|  2020-03-30| 1585526537|    3|           3| frgabe|    5|I fell like this ...|archive|depression|It always gets be...|2020|
|  2020-03-30| 1585526611|    3|           0| frgb0r|    1|           [removed]|archive|depression|Grew up in a Roug...|2020|
|  2020-03-30| 1585526614|    3|           0| frgb1z|    2|I'd prefer if u d...|archive|depression|Need help Trying ...|2020|
|  2020-03-30| 1585526865|    3|           1| frgdei|    2|           [deleted]|archive|depression|    Two options left|2020|
|  2020-03-30| 1585526978|    3|           3| frgeiy|    3|I'm a little tips...|archive|depression|                Hop

In [8]:
comments.show(5, truncate=True)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+----------+------------+-----------+-----+-------+-----+-------+----------+----+
|                body|comment_id|created_date|created_utc|month|post_id|score| source| subreddit|year|
+--------------------+----------+------------+-----------+-----+-------+-----+-------+----------+----+
|Oh, don't worry, ...|   flvlho3|  2020-03-30| 1585526440|    3| freus2|    2|archive|depression|2020|
|I usually try to ...|   flvliov|  2020-03-30| 1585526459|    3| fr948o|    1|archive|depression|2020|
|This makes me so ...|   flvlk2m|  2020-03-30| 1585526484|    3| frdkuf|    1|archive|depression|2020|
|I think I need a ...|   flvlk68|  2020-03-30| 1585526486|    3| fr95po|    3|archive|depression|2020|
|Thank you. Just t...|   flvll0f|  2020-03-30| 1585526500|    3| frey9o|    2|archive|depression|2020|
+--------------------+----------+------------+-----------+-----+-------+-----+-------+----------+----+
only showing top 5 rows

# 2 Data Cleaning

In [9]:
posts_clean = posts.filter(
    (col("selftext").isNotNull()) &
    (col("selftext") != "[deleted]") &
    (col("selftext") != "[removed]") &
    (trim(col("selftext")) != "")
)

comments_clean = comments.filter(
    (col("body").isNotNull()) &
    (col("body") != "[deleted]") &
    (col("body") != "[removed]") &
    (trim(col("body")) != "")
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
posts_clean.cache()
comments_clean.cache()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[body: string, comment_id: string, created_date: string, created_utc: bigint, month: int, post_id: string, score: bigint, source: string, subreddit: string, year: int]

In [11]:
print("Original posts:", posts.count())
print("Clean posts:", posts_clean.count())

print("Original comments:", comments.count())
print("Clean comments:", comments_clean.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Original posts: 1175
Clean posts: 728
Original comments: 3810
Clean comments: 3810

# 3. Feature engineering

In [12]:
tokenizer = Tokenizer(
    inputCol="clean_text",
    outputCol="tokens"
)

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
# Posts NLP
posts_nlp = posts_clean.withColumn(
    "clean_text",
    lower(col("selftext"))
)

posts_nlp = posts_nlp.withColumn(
    "clean_text",
    regexp_replace("clean_text", "[^a-zA-Z\\s]", "")
)

posts_nlp = tokenizer.transform(posts_nlp)

posts_nlp = remover.transform(posts_nlp)

posts_nlp = posts_nlp.withColumn(
    "filtered_tokens",
    array_remove("filtered_tokens", "")
)

posts_nlp = posts_nlp.withColumn(
    "num_tokens",
    size("filtered_tokens")
)

posts_nlp.cache()

posts_nlp.select(
    "clean_text",
    "tokens",
    "filtered_tokens"
).show(5, truncate=50)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+
|                                        clean_text|                                            tokens|                                   filtered_tokens|
+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+
|i fell like this is said a lot and to some exte...|[i, fell, like, this, is, said, a, lot, and, to...|[fell, like, said, lot, extent, true, time, alw...|
|id prefer if u dm instead of comment cuz commen...|[id, prefer, if, u, dm, instead, of, comment, c...|[id, prefer, u, dm, instead, comment, cuz, comm...|
|im a little tipsy and just a smidge high on the...|[im, a, little, tipsy, and, just, a, smidge, hi...|[im, little, tipsy, smidge, high, reefer, feel,...|
|live with my parents im disabled no irl or onli...|[live, with, my, p

In [14]:
# Comments NLP
comments_nlp = comments_clean.withColumn(
    "clean_text",
    lower(col("body"))
)

comments_nlp = comments_nlp.withColumn(
    "clean_text",
    regexp_replace("clean_text", "[^a-zA-Z\\s]", "")
)

comments_nlp = tokenizer.transform(comments_nlp)

comments_nlp = remover.transform(comments_nlp)

comments_nlp = comments_nlp.withColumn(
    "filtered_tokens",
    array_remove("filtered_tokens", "")
)

comments_nlp = comments_nlp.withColumn(
    "num_tokens",
    size("filtered_tokens")
)

comments_nlp.cache()

comments_nlp.select(
    "clean_text",
    "tokens",
    "filtered_tokens"
).show(5, truncate=50)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+
|                                        clean_text|                                            tokens|                                   filtered_tokens|
+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+
|oh dont worry you will certainly not find love ...|[oh, dont, worry, you, will, certainly, not, fi...|[oh, dont, worry, certainly, find, love, state,...|
|i usually try to tell myself theres always tomo...|[i, usually, try, to, tell, myself, theres, alw...|[usually, try, tell, theres, always, tomorrow, ...|
|this makes me so sad i promise youre not worthl...|[this, makes, me, so, sad, i, promise, youre, n...|[makes, sad, promise, youre, worthless, informa...|
|i think i need a few more sessions before i can...|[i, think, i, need

# 4. Word Frequency Analysis

In [15]:
posts_words = posts_nlp.select(
    explode("filtered_tokens").alias("word")
)

posts_words.groupBy("word").count().orderBy(
    col("count").desc()
).show(20)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+-----+
|   word|count|
+-------+-----+
|     im| 1474|
|   like|  877|
|   dont|  832|
|   feel|  693|
|   know|  597|
|    ive|  510|
|   even|  492|
|    get|  489|
|   want|  464|
| really|  463|
|   time|  432|
|   cant|  401|
|   life|  363|
| people|  360|
|    one|  352|
|  think|  322|
|anxiety|  307|
|  going|  283|
|  never|  278|
| things|  277|
+-------+-----+
only showing top 20 rows

In [16]:
comments_words = comments_nlp.select(
    explode("filtered_tokens").alias("word")
)

comments_words.groupBy("word").count().orderBy(
    col("count").desc()
).show(20)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+-----+
|   word|count|
+-------+-----+
|     im| 1476|
|   like| 1199|
|   dont| 1042|
|    get|  902|
|   feel|  849|
| people|  840|
|   know|  751|
| really|  651|
|   time|  636|
|  think|  602|
|anxiety|  535|
|   help|  508|
| things|  507|
|  youre|  489|
|    one|  478|
|   even|  469|
|    ive|  456|
|   life|  440|
|  going|  428|
|     go|  405|
+-------+-----+
only showing top 20 rows

# 5. Monthly Aggregation

In [17]:
monthly_posts = posts_nlp.groupBy(
    "year",
    "month",
    "subreddit"
).count().orderBy(
    "year",
    "month"
)

monthly_posts.show()

monthly_comments = comments_nlp.groupBy(
    "year",
    "month",
    "subreddit"
).count().orderBy(
    "year",
    "month"
)

monthly_comments.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+-----+------------+-----+
|year|month|   subreddit|count|
+----+-----+------------+-----+
|2020|    3|mentalhealth|  180|
|2020|    3|  depression|  360|
|2020|    3|     anxiety|  188|
+----+-----+------------+-----+

+----+-----+------------+-----+
|year|month|   subreddit|count|
+----+-----+------------+-----+
|2020|    3|     anxiety| 1492|
|2020|    3|  depression| 1575|
|2020|    3|mentalhealth|  743|
+----+-----+------------+-----+

# 6. Average Token Counts

In [18]:
posts_nlp.groupBy(
    "year",
    "month"
).avg("num_tokens").show()

comments_nlp.groupBy(
    "year",
    "month"
).avg("num_tokens").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+-----+-----------------+
|year|month|  avg(num_tokens)|
+----+-----+-----------------+
|2020|    3|86.36126373626374|
+----+-----+-----------------+

+----+-----+------------------+
|year|month|   avg(num_tokens)|
+----+-----+------------------+
|2020|    3|25.146456692913386|
+----+-----+------------------+

# 7. Save Processed Data as Parquet

In [19]:
posts_nlp.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet(
        "s3://nana-survey-bucket-2026/reddit/posts_nlp/"
    )

comments_nlp.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet(
        "s3://nana-survey-bucket-2026/reddit/comments_nlp/"
    )

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# 8. Scalability Benchmark

In [20]:
# Fixed random seed for reproducibility
SEED = 42

fractions = [0.1, 0.5, 1.0]

for frac in fractions:

    print(f"\n Benchmark: {int(frac*100)}% Data ")

    start = time.time()

    # Sample Data

    sample_posts = posts.sample(
        withReplacement=False,
        fraction=frac,
        seed=SEED
    )

    # Cleaning

    sample_clean = sample_posts.filter(
        (col("selftext").isNotNull()) &
        (col("selftext") != "[deleted]") &
        (col("selftext") != "[removed]") &
        (trim(col("selftext")) != "")
    )

    # NLP preprocessing

    sample_nlp = sample_clean.withColumn(
        "clean_text",
        lower(col("selftext"))
    )

    sample_nlp = sample_nlp.withColumn(
        "clean_text",
        regexp_replace(
            "clean_text",
            "[^a-zA-Z\\s]",
            ""
        )
    )

    sample_nlp = tokenizer.transform(sample_nlp)

    sample_nlp = remover.transform(sample_nlp)

    # Aggregation

    sample_monthly = sample_nlp.groupBy(
        "year",
        "month"
    ).count()

    # Force execution
    sample_monthly.count()

    end = time.time()

    runtime = end - start

    print(f"Runtime: {runtime:.2f} seconds")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


 Benchmark: 10% Data 
1
Runtime: 0.66 seconds

 Benchmark: 50% Data 
1
Runtime: 0.50 seconds

 Benchmark: 100% Data 
1
Runtime: 0.50 seconds